# OceanWatch — Entrega 1 · Requisitos 3 y 4

**Requisito 3 — Preguntas de negocio (A-E)** respondidas con Spark sobre los CSV del Volume,
justificando las decisiones con el plan de ejecución.

**Requisito 4 — Almacenamiento óptimo para un propósito:** se materializa la tabla Delta
y se demuestra la mejora frente a CSV y Parquet con bytes, archivos leídos y efecto de
`OPTIMIZE`.

Prerrequisito: haber corrido `01_ingesta_perfilamiento`, que deja los 7 CSV en
`/Volumes/mine4213/proyecto/data/csv`.

## Imports y carga

In [ ]:
import os
import time

from pyspark.sql import Window
from pyspark.sql import functions as f

CATALOG = "mine4213"
SCHEMA = "proyecto"
VOLUME = "data"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""


def leer_csv():
    return (
        spark.read
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(AIS_SCHEMA)
        .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    )


ais = (
    leer_csv()
    .withColumn("fecha", f.to_date("BaseDateTime"))
)

ais.printSchema()

# Requisito 3 — Preguntas de negocio

## Bases analíticas mínimas

In [ ]:
mmsi_valido = (
    f.col("MMSI").isNotNull()
    & f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")
)

coordenada_valida = (
    f.col("LAT").between(-90, 90)
    & f.col("LON").between(-180, 180)
)

ais_buques = (
    ais
    .filter(mmsi_valido)
)

ais_espacial = (
    ais
    .filter(coordenada_valida)
)

## A. ¿Cuántos buques distintos transmitieron cada día?

Se compara el resultado exacto obtenido mediante `countDistinct` con
`approx_count_distinct`.

Para esta pregunta se consideran únicamente MMSI con formato válido de nueve
dígitos, ya que el perfilamiento identificó identificadores anómalos que no
deben interpretarse automáticamente como buques distintos.

### Conteo exacto

In [ ]:
exactos = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.countDistinct("MMSI").alias("buques_exactos")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO EXACTO")
exactos.explain("formatted")

In [ ]:
inicio = time.perf_counter()

exact_rows = exactos.collect()

tiempo_exacto = time.perf_counter() - inicio

print(f"Tiempo conteo exacto: {tiempo_exacto:.2f} segundos")

### Conteo aproximado

In [ ]:
aproximados = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.approx_count_distinct("MMSI").alias("buques_aproximados")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO APROXIMADO")
aproximados.explain("formatted")

In [ ]:
inicio = time.perf_counter()

approx_rows = aproximados.collect()

tiempo_aproximado = time.perf_counter() - inicio

print(f"Tiempo conteo aproximado: {tiempo_aproximado:.2f} segundos")

### Comparación

In [ ]:
exact_map = {
    row["fecha"]: row["buques_exactos"]
    for row in exact_rows
}

approx_map = {
    row["fecha"]: row["buques_aproximados"]
    for row in approx_rows
}

comparacion_data = []

for fecha in sorted(exact_map.keys()):

    exacto = exact_map[fecha]
    aproximado = approx_map[fecha]

    error_abs = abs(aproximado - exacto)

    error_pct = (
        100 * error_abs / exacto
        if exacto > 0
        else 0
    )

    comparacion_data.append(
        (
            fecha,
            exacto,
            aproximado,
            error_abs,
            float(error_pct)
        )
    )

comparacion_a = spark.createDataFrame(
    comparacion_data,
    [
        "fecha",
        "buques_exactos",
        "buques_aproximados",
        "error_absoluto",
        "error_porcentual"
    ]
)

display(comparacion_a)

display(
    comparacion_a
    .agg(
        f.round(
            f.avg("error_porcentual"),
            4
        ).alias("error_porcentual_medio"),

        f.round(
            f.max("error_porcentual"),
            4
        ).alias("error_porcentual_maximo")
    )
)

## B. ¿Qué tipos de buque generan más tráfico?

Se calcula el Top 10 de códigos de tipo de buque según el número de posiciones
AIS transmitidas durante la semana.

Los códigos se enriquecen utilizando el catálogo oficial de VesselType de
Marine Cadastre. [catalogo](https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf)

Para la velocidad media se excluye SOG=102.3 porque el perfilamiento determinó
que representa velocidad no disponible y no una velocidad real.

### Catálogo de tipos

In [ ]:
catalogo = []

def agregar(codigo, grupo, descripcion):
    catalogo.append(
        (codigo, grupo, descripcion)
    )


# 0
agregar(
    0,
    "Not Available",
    "Not available or no ship, default"
)

# 1-19
for c in range(1, 20):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )


# 20-29: WIG
wig = {
    20: ("Other", "Wing in ground (WIG), all ships of this type"),
    21: ("Tug Tow", "Wing in ground (WIG), hazardous category A"),
    22: ("Tug Tow", "Wing in ground (WIG), hazardous category B"),
    23: ("Other", "Wing in ground (WIG), hazardous category C"),
    24: ("Other", "Wing in ground (WIG), hazardous category D"),
    25: ("Other", "Wing in ground (WIG), reserved for future use"),
    26: ("Other", "Wing in ground (WIG), reserved for future use"),
    27: ("Other", "Wing in ground (WIG), reserved for future use"),
    28: ("Other", "Wing in ground (WIG), reserved for future use"),
    29: ("Other", "Wing in ground (WIG), reserved for future use"),
}

for codigo, (grupo, descripcion) in wig.items():
    agregar(codigo, grupo, descripcion)


# 30-59
tipos_especiales = {
    30: ("Fishing", "Fishing"),
    31: ("Tug Tow", "Towing"),
    32: ("Tug Tow", "Towing: length exceeds 200m or breadth exceeds 25m"),
    33: ("Other", "Dredging or underwater operations"),
    34: ("Other", "Diving operations"),
    35: ("Military", "Military operations"),
    36: ("Pleasure Craft/Sailing", "Sailing"),
    37: ("Pleasure Craft/Sailing", "Pleasure Craft"),
    38: ("Other", "Reserved"),
    39: ("Other", "Reserved"),

    40: ("Other", "High speed craft (HSC), all ships of this type"),
    41: ("Other", "High speed craft (HSC), hazardous category A"),
    42: ("Other", "High speed craft (HSC), hazardous category B"),
    43: ("Other", "High speed craft (HSC), hazardous category C"),
    44: ("Other", "High speed craft (HSC), hazardous category D"),
    45: ("Other", "High speed craft (HSC), reserved for future use"),
    46: ("Other", "High speed craft (HSC), reserved for future use"),
    47: ("Other", "High speed craft (HSC), reserved for future use"),
    48: ("Other", "High speed craft (HSC), reserved for future use"),
    49: ("Other", "High speed craft (HSC), no additional information"),

    50: ("Other", "Pilot Vessel"),
    51: ("Other", "Search and Rescue vessel"),
    52: ("Tug Tow", "Tug"),
    53: ("Other", "Port Tender"),
    54: ("Other", "Anti-pollution equipment"),
    55: ("Other", "Law Enforcement"),
    56: ("Other", "Spare - for assignment to local vessel"),
    57: ("Other", "Spare - for assignment to local vessel"),
    58: ("Other", "Medical Transport"),
    59: ("Other", "Ship according to RR Resolution No. 18"),
}

for codigo, (grupo, descripcion) in tipos_especiales.items():
    agregar(codigo, grupo, descripcion)

In [ ]:
familias = {
    60: ("Passenger", "Passenger"),
    70: ("Cargo", "Cargo"),
    80: ("Tanker", "Tanker"),
    90: ("Other", "Other Type")
}

sufijos = {
    0: "all ships of this type",
    1: "hazardous category A",
    2: "hazardous category B",
    3: "hazardous category C",
    4: "hazardous category D",
    5: "reserved for future use",
    6: "reserved for future use",
    7: "reserved for future use",
    8: "reserved for future use",
    9: "no additional information"
}

for base, (grupo, nombre) in familias.items():

    for offset in range(10):

        codigo = base + offset

        agregar(
            codigo,
            grupo,
            f"{nombre}, {sufijos[offset]}"
        )

In [ ]:
for c in range(100, 200):
    agregar(
        c,
        "Other",
        "Reserved for regional use"
    )

for c in range(200, 256):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )

for c in range(256, 1000):
    agregar(
        c,
        "Other",
        "No designation"
    )

In [ ]:
avis = {
    1001: ("Fishing", "Commercial Fishing Vessel"),
    1002: ("Fishing", "Fish Processing Vessel"),
    1003: ("Cargo", "Freight Barge"),
    1004: ("Cargo", "Freight Ship"),
    1005: ("Other", "Industrial Vessel"),
    1006: ("Other", "Miscellaneous Vessel"),
    1007: ("Other", "Mobile Offshore Drilling Unit"),
    1008: ("Other", "Non-vessel"),
    1009: ("Other", "NON-VESSEL"),
    1010: ("Other", "Offshore Supply Vessel"),
    1011: ("Other", "Oil Recovery"),
    1012: ("Passenger", "Passenger (Inspected)"),
    1013: ("Passenger", "Passenger (Uninspected)"),
    1014: ("Passenger", "Passenger Barge (Inspected)"),
    1015: ("Passenger", "Passenger Barge (Uninspected)"),
    1016: ("Cargo", "Public Freight"),
    1017: ("Tanker", "Public Tankship/Barge"),
    1018: ("Other", "Public Vessel, Unclassified"),
    1019: ("Pleasure Craft/Sailing", "Recreational"),
    1020: ("Other", "Research Vessel"),
    1021: ("Military", "SAR Aircraft"),
    1022: ("Other", "School Ship"),
    1023: ("Tug Tow", "Tank Barge"),
    1024: ("Tanker", "Tank Ship"),
    1025: ("Tug Tow", "Towing Vessel")
}

for codigo, (grupo, descripcion) in avis.items():
    agregar(
        codigo,
        grupo,
        descripcion
    )

In [ ]:
catalogo_tipos = spark.createDataFrame(
    catalogo,
    [
        "VesselType",
        "grupo_buque",
        "descripcion_tipo"
    ]
)

display(catalogo_tipos.limit(20))

### Respuesta

In [ ]:
ais_velocidad = (
    ais
    .withColumn(
        "SOG_utilizable",
        f.when(
            f.col("SOG").between(0, 102.2),
            f.col("SOG")
        )
    )
)

In [ ]:
top10_tipos_base = (
    ais_velocidad
    .filter(
        f.col("VesselType").isNotNull()
    )
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("numero_posiciones"),

        f.round(
            f.avg("SOG_utilizable"),
            3
        ).alias("velocidad_media_nudos"),

        f.count("SOG_utilizable").alias(
            "posiciones_con_velocidad_utilizable"
        )
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
    .limit(10)
)

In [ ]:
resultado_b = (
    top10_tipos_base
    .join(
        f.broadcast(catalogo_tipos),
        on="VesselType",
        how="left"
    )
    .select(
        "VesselType",
        "grupo_buque",
        "descripcion_tipo",
        "numero_posiciones",
        "velocidad_media_nudos",
        "posiciones_con_velocidad_utilizable"
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
)

display(resultado_b)

In [ ]:
resultado_b.explain("formatted")

## C. ¿Qué 10 buques recorrieron más distancia durante la semana?

In [ ]:
window_vessel = Window.partitionBy("MMSI").orderBy("BaseDateTime")

df_lag = (ais
    .withColumn("LAT_prev", f.lag("LAT").over(window_vessel))
    .withColumn("LON_prev", f.lag("LON").over(window_vessel))
    .filter(f.col("LAT_prev").isNotNull() & f.col("LON_prev").isNotNull())
)

lat1 = f.radians(f.col("LAT_prev"))
lon1 = f.radians(f.col("LON_prev"))
lat2 = f.radians(f.col("LAT"))
lon2 = f.radians(f.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

R_NM = 3440.0654

a = (f.sin(dlat / 2) ** 2) + f.cos(lat1) * f.cos(lat2) * (f.sin(dlon / 2) ** 2)
c = 2 * f.atan2(f.sqrt(a), f.sqrt(1 - a))
distancia_tramo = R_NM * c

df_distancias = (df_lag
    .withColumn("distancia_nm", distancia_tramo)
    .filter(f.col("distancia_nm") < 100)
)

top10_distancia = (df_distancias
    .groupBy("MMSI", "VesselName")
    .agg(
        f.round(f.sum("distancia_nm"), 2).alias("distancia_total_millas_nauticas"),
        f.round(f.try_divide(f.sum("distancia_nm"), f.avg("SOG")),2).alias("tiempo_total_horas")
    )
    .orderBy(f.col("distancia_total_millas_nauticas").desc())
    .limit(10)
)

display(top10_distancia)

In [ ]:
top10_distancia.explain("formatted")

## D. ¿Dónde se concentra el tráfico?

In [ ]:
df_h3 = ais.withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))

cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("num_posiciones"),
        f.round(f.avg("LAT"), 4).alias("lat_centroide"),
        f.round(f.avg("LON"), 4).alias("lon_centroide")
    )
    .orderBy(f.col("num_posiciones").desc())
    .limit(10)
)

display(cells)

In [ ]:
cells.explain("formatted")

Falta lo de puertos

## E. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

### 1. Proporción de buques que transmitieron los 7 días

In [ ]:
df_dias_actividad = (ais
    .groupBy("MMSI")
    .agg(f.countDistinct("fecha").alias("dias_activos"))
)

proporcion_7_dias = (df_dias_actividad
    .select(
        f.count("MMSI").alias("total_buques"),
        f.sum(f.when(f.col("dias_activos") == 7, 1).otherwise(0)).alias("buques_7_dias"),
        f.sum(f.when(f.col("dias_activos") == 1, 1).otherwise(0)).alias("buques_1_dia")
    )
    .withColumn("porcentaje_7_dias", f.round((f.col("buques_7_dias") / f.col("total_buques")) * 100, 2))
    .withColumn("porcentaje_1_dia", f.round((f.col("buques_1_dia") / f.col("total_buques")) * 100, 2))
)

display(proporcion_7_dias)

La proporción de buques que vistaron los 7 días es: $$ \frac{12667}{31871} $$

Lo cual representa un 39.74% de todos los buques.

### 2. Ubicación de los visitantes de un solo día

In [ ]:
mmsi_visitantes_1_dia = df_dias_actividad.filter(f.col("dias_activos") == 1).select("MMSI")

df_visitantes = ais.join(mmsi_visitantes_1_dia, on="MMSI", how="inner")

display(df_visitantes.limit(100))

In [ ]:
ubicacion_visitantes = (df_visitantes
    .withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("num_buques_visitantes"),
        f.round(f.avg("LAT"), 4).alias("lat_promedio"),
        f.round(f.avg("LON"), 4).alias("lon_promedio")
    )
    .orderBy(f.col("num_buques_visitantes").desc())
)

display(ubicacion_visitantes)

In [ ]:
ubicacion_visitantes.explain("formatted")

# Requisito 4 — Almacenamiento óptimo para un propósito

Se sigue la estructura de decisiones de la semana 6 (formato de archivo → formato de
tabla → layout → mantenimiento) y su rúbrica: **resultado + evidencia** — tamaños en
disco, el `_delta_log` y planes o perfiles de consulta. **No se usan tiempos**: en
serverless no discriminan.

## 4.1 Propósito de consulta

**La consulta diaria del operador portuario:** para un día y una zona marítima
(bounding box lat/lon), cuántas posiciones y cuántos buques distintos hubo por hora.

Ejemplo: zona de Houston / Galveston el 3 de junio. La consulta:

- lee **4 de las 17 columnas** (`BaseDateTime`, `LAT`, `LON`, `MMSI`, más `fecha`);
- filtra por **fecha** (igualdad) y por **LAT/LON** (rangos).

Cada propiedad obliga a una decisión: columnas → formato de archivo (4.2); filtros →
layout (4.4).

In [ ]:
import json

FECHA_CONSULTA = "2023-06-03"

LAT_MIN, LAT_MAX = 29.0, 30.0
LON_MIN, LON_MAX = -95.5, -94.5

LAB = f"{BASE}/almacenamiento"

RUTA_PARQUET = f"{LAB}/ais_parquet"
RUTA_DELTA = f"{LAB}/ais_delta"
RUTA_PARTICIONADA = f"{LAB}/ais_delta_particionada"
RUTA_CLUSTER = f"{LAB}/ais_delta_cluster"

TABLA_FINAL = f"{CATALOG}.{SCHEMA}.ais_posiciones"

CODECS = ["snappy", "gzip", "zstd", "lz4"]


def peso(ruta):
    """Bytes totales de una ruta, recursivo."""
    total = 0
    for p in dbutils.fs.ls(ruta):
        total += peso(p.path) if p.isDir() else p.size
    return total


def humano(n):
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"


def filtro_proposito(fecha_col):
    return (
        (fecha_col == f.lit(FECHA_CONSULTA).cast("date"))
        & f.col("LAT").between(LAT_MIN, LAT_MAX)
        & f.col("LON").between(LON_MIN, LON_MAX)
    )


def consulta_proposito(df, fecha_col=f.col("fecha")):
    return (
        df
        .filter(filtro_proposito(fecha_col))
        .groupBy(f.hour("BaseDateTime").alias("hora"))
        .agg(
            f.count("*").alias("posiciones"),
            f.countDistinct("MMSI").alias("buques")
        )
        .orderBy("hora")
    )

In [ ]:
# ──── reset: el laboratorio de almacenamiento parte de cero en cada corrida ────
spark.sql(f"DROP TABLE IF EXISTS {TABLA_FINAL}")
dbutils.fs.rm(LAB, True)
dbutils.fs.mkdirs(LAB)

print(f"{LAB} limpio. Los CSV de {CSV_DIR} no se tocan.")

## 4.2 Decisión 1 — Formato de archivo: CSV vs Parquet (y codec)

Parquet es columnar: la consulta abre solo las columnas que pide y cada columna se
comprime por separado. El codec es un triángulo: snappy (rápido), gzip (compacto),
zstd (el equilibrio), lz4.

In [ ]:
csv_bytes = peso(CSV_DIR)

for codec in CODECS:
    (
        ais
        .write
        .mode("overwrite")
        .option("compression", codec)
        .parquet(f"{RUTA_PARQUET}_{codec}")
    )

formatos = [("CSV (texto)", csv_bytes)] + [
    (f"Parquet {codec}", peso(f"{RUTA_PARQUET}_{codec}"))
    for codec in CODECS
]

display(
    spark.createDataFrame(formatos, ["formato", "bytes"])
    .withColumn("GB", f.round(f.col("bytes") / 1024**3, 3))
    .withColumn("veces_menor_que_csv", f.round(f.lit(csv_bytes) / f.col("bytes"), 1))
)

**Elegir el codec** con la tabla de arriba y fijarlo en `CODEC`: se usa para todas las
tablas Delta de aquí en adelante.

In [ ]:
CODEC = "zstd"

RUTA_PARQUET_ELEGIDO = f"{RUTA_PARQUET}_{CODEC}"

### Columnas leídas: la consulta del propósito, dos caminos

En el plan, el `ReadSchema` del scan Parquet trae solo las columnas de la consulta; el
CSV tiene que parsear las 17 de cada fila. Correr los `display` y abrir el **query
profile** (ícono bajo el resultado): bytes y columnas leídas.

In [ ]:
por_csv = consulta_proposito(leer_csv(), f.to_date("BaseDateTime"))

por_csv.explain("formatted")

display(por_csv)

In [ ]:
por_parquet = consulta_proposito(spark.read.parquet(RUTA_PARQUET_ELEGIDO))

por_parquet.explain("formatted")

display(por_parquet)

## 4.3 Decisión 2 — Formato de tabla: Parquet vs Delta

Delta = los mismos Parquet + un `_delta_log`. El log es lo que habilita:

- **estadísticas min/max por archivo**, que permiten saltar archivos (4.4);
- `OPTIMIZE` y liquid clustering (4.4, 4.5);
- ACID, `DESCRIBE HISTORY` y time travel: la Entrega 2 va a limpiar y corregir estos
  datos (duplicados, sentinels, MMSI inválidos) y cada corrección queda auditada.

In [ ]:
(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .save(RUTA_DELTA)
)

print(f"Parquet {CODEC}:         {humano(peso(RUTA_PARQUET_ELEGIDO))}")
print(f"Delta ({CODEC}, sin layout): {humano(peso(RUTA_DELTA))}")

In [ ]:
!ls -lnh $RUTA_DELTA | head -20

In [ ]:
!ls -lnh $RUTA_DELTA/_delta_log

### Lo que guarda el log por archivo

Cada acción `add` del log registra un archivo Parquet con su tamaño y sus estadísticas
(`numRecords`, `minValues`, `maxValues`). Con esas estadísticas se reconstruye qué
archivos **podrían** tener filas del filtro: el resto se salta sin abrirlo.

In [ ]:
def commits(ruta):
    """Commits JSON del _delta_log, ascendentes: [(número, [acciones])]."""
    log = f"{ruta.rstrip('/')}/_delta_log"
    salida = []
    for nombre in sorted(os.listdir(log)):
        if nombre.endswith(".json"):
            with open(f"{log}/{nombre}", encoding="utf-8") as archivo:
                acciones = [json.loads(linea) for linea in archivo if linea.strip()]
            salida.append((int(nombre.split(".")[0]), acciones))
    return salida


def archivos_activos(ruta):
    """Acciones add vigentes en la última versión (add menos remove)."""
    activos = {}
    for _, acciones in commits(ruta):
        for accion in acciones:
            if "add" in accion:
                activos[accion["add"]["path"]] = accion["add"]
            elif "remove" in accion:
                activos.pop(accion["remove"]["path"], None)
    return list(activos.values())


def podria_tener_filas(add):
    """Data skipping: ¿el rango del archivo intersecta el filtro del propósito?"""
    particion = add.get("partitionValues") or {}
    stats = json.loads(add.get("stats") or "{}")
    minimos = stats.get("minValues", {})
    maximos = stats.get("maxValues", {})

    def intersecta(col, bajo, alto):
        if col in particion:
            return bajo <= particion[col] <= alto
        if col not in minimos or col not in maximos:
            return True  # sin estadísticas no se puede saltar
        return not (maximos[col] < bajo or minimos[col] > alto)

    return (
        intersecta("fecha", FECHA_CONSULTA, FECHA_CONSULTA)
        and intersecta("LAT", LAT_MIN, LAT_MAX)
        and intersecta("LON", LON_MIN, LON_MAX)
    )


def evidencia_layout(nombre, ruta):
    activos = archivos_activos(ruta)
    candidatos = [add for add in activos if podria_tener_filas(add)]
    return (
        nombre,
        len(activos),
        len(candidatos),
        sum(add["size"] for add in activos),
        sum(add["size"] for add in candidatos),
    )


ejemplo = archivos_activos(RUTA_DELTA)[0]
print(ejemplo["path"])
print(json.dumps(json.loads(ejemplo["stats"]), indent=2)[:1500])

## 4.4 Decisión 3 — Layout: ¿dónde vive cada fila?

Tres generaciones del layout con la misma data y el mismo tamaño objetivo de archivo
(`delta.targetFileSize = 32m`, como en la semana 6, para que haya granularidad suficiente
para saltar archivos):

| Layout | Qué poda |
|---|---|
| sin layout | solo lo que el orden de llegada deje por azar |
| `partitionBy("fecha")` | días completos (carpetas `fecha=...`); nada por zona |
| `clusterBy("fecha", "LAT", "LON")` | fecha **y** zona: cada archivo cubre un rango acotado de las tres columnas |

Particionar también por zona no es opción: miles de combinaciones fecha × zona con
pocos datos cada una (el problema de archivos pequeños).

In [ ]:
(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .partitionBy("fecha")
    .save(RUTA_PARTICIONADA)
)

(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .clusterBy("fecha", "LAT", "LON")
    .save(RUTA_CLUSTER)
)

In [ ]:
!ls -lnh $RUTA_PARTICIONADA

In [ ]:
!ls -lnh $RUTA_PARTICIONADA/fecha=2023-06-03 | head -20

In [ ]:
!ls -lnh $RUTA_CLUSTER | head -20

### Evidencia: archivos que la consulta del propósito tiene que abrir

Calculado desde el `_delta_log` de cada tabla con las estadísticas por archivo
(misma lógica que usa Delta para el data skipping). Se contrasta con el query profile
de las celdas siguientes (*files read* / *files pruned*).

In [ ]:
layouts = [
    ("delta sin layout", RUTA_DELTA),
    ("delta partitionBy(fecha)", RUTA_PARTICIONADA),
    ("delta clusterBy(fecha, LAT, LON)", RUTA_CLUSTER),
]

evidencia_df = (
    spark.createDataFrame(
        [evidencia_layout(nombre, ruta) for nombre, ruta in layouts],
        [
            "layout",
            "archivos_totales",
            "archivos_a_leer",
            "bytes_totales",
            "bytes_a_leer"
        ]
    )
    .withColumn(
        "porcentaje_archivos_saltados",
        f.round(100 * (1 - f.col("archivos_a_leer") / f.col("archivos_totales")), 2)
    )
    .withColumn("MB_a_leer", f.round(f.col("bytes_a_leer") / 1024**2, 1))
)

display(evidencia_df)

### Planes de ejecución

En la particionada el filtro de fecha aparece como `PartitionFilters`; en la
clusterizada, fecha y LAT/LON quedan como `DataFilters` que Delta usa contra las
estadísticas del log. Correr los `display` y comparar en el query profile los
archivos leídos vs. podados.

In [ ]:
por_particion = consulta_proposito(spark.read.format("delta").load(RUTA_PARTICIONADA))

por_particion.explain("formatted")

display(por_particion)

In [ ]:
por_cluster = consulta_proposito(spark.read.format("delta").load(RUTA_CLUSTER))

por_cluster.explain("formatted")

display(por_cluster)

## 4.5 Mantenimiento: efecto de OPTIMIZE

`OPTIMIZE` compacta archivos pequeños y, en una tabla con liquid clustering, reagrupa los
datos por las claves de clustering. Escribe archivos nuevos y marca los viejos como
`remove` en el log, **pero no los borra del disco**: eso lo hace `VACUUM` pasado el
periodo de retención (7 días por defecto), para no romper el time travel.

Nota: en serverless las escrituras ya vienen con *optimized writes*, así que el efecto
puede ser pequeño. `DESCRIBE HISTORY` muestra qué hizo exactamente.

In [ ]:
antes = {
    nombre: (evidencia_layout(nombre, ruta), peso(ruta))
    for nombre, ruta in layouts
}

for _, ruta in layouts:
    display(spark.sql(f"OPTIMIZE delta.`{ruta}`"))

despues = {
    nombre: (evidencia_layout(nombre, ruta), peso(ruta))
    for nombre, ruta in layouts
}

In [ ]:
efecto_optimize = spark.createDataFrame(
    [
        (
            nombre,
            antes[nombre][0][1],
            despues[nombre][0][1],
            antes[nombre][0][2],
            despues[nombre][0][2],
            antes[nombre][1],
            despues[nombre][1],
        )
        for nombre, _ in layouts
    ],
    [
        "layout",
        "archivos_activos_antes",
        "archivos_activos_despues",
        "archivos_a_leer_antes",
        "archivos_a_leer_despues",
        "bytes_en_disco_antes",
        "bytes_en_disco_despues"
    ]
)

display(efecto_optimize)

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY delta.`{RUTA_CLUSTER}`")
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
)

Los bytes en disco **suben** después de `OPTIMIZE`: los archivos compactados conviven con
los viejos hasta un `VACUUM`. Los archivos *activos* (los que lee una consulta) son los
que bajan.

## 4.6 Tabla final en Unity Catalog

Con la decisión tomada, la tabla del proyecto se materializa como tabla administrada en
`mine4213.proyecto`, con el mismo formato, codec y layout, y con comentario.

In [ ]:
(
    spark.read.format("delta").load(RUTA_CLUSTER)
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .clusterBy("fecha", "LAT", "LON")
    .saveAsTable(TABLA_FINAL)
)

spark.sql(f"""
    COMMENT ON TABLE {TABLA_FINAL} IS
    'Posiciones AIS de NOAA, 1-7 junio 2023, sin limpiar. Liquid clustering por (fecha, LAT, LON) para la consulta diaria del operador por fecha y zona marítima.'
""")

display(spark.sql(f"DESCRIBE DETAIL {TABLA_FINAL}"))

## 4.7 Decisión

_Completar con los números obtenidos arriba._

- **Formato de archivo:** Parquet `CODEC` — X veces menos bytes que el CSV (4.2) y el
  scan lee solo las columnas de la consulta (`ReadSchema` del plan).
- **Formato de tabla:** Delta — mismos bytes que Parquet + `_delta_log` con estadísticas
  por archivo, `OPTIMIZE`, clustering e historial para las correcciones de la Entrega 2.
- **Layout:** `clusterBy(fecha, LAT, LON)` — la consulta del propósito abre N de M
  archivos, contra N' con `partitionBy(fecha)` y M sin layout (4.4).
- **Mantenimiento:** efecto de `OPTIMIZE` según 4.5; `VACUUM` pendiente cuando pase el
  periodo de retención.